# Classer des intentions avec un LLM, et comparer

Hier, les modèles classiques, les *embeddings* et un encodeur affiné ont classé les demandes de MASSIVE. Aujourd'hui, un LLM, sans entraînement :

1. Sans exemple : la liste des intentions décrites, une sortie contrainte
2. Avec quelques exemples dans le prompt
3. Comparer toutes les méthodes
4. Lire les erreurs

Ce notebook a besoin d'un GPU : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données et modèle

Les résultats d'hier : chargez vos fichiers `predictions-fr.csv` et `timings-fr.csv` (panneau *Fichiers* à gauche), ou utilisez les nôtres, téléchargés par la cellule suivante s'ils manquent. Ollama s'installe et télécharge le modèle en arrière-plan.

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s massive-intents
!test -f predictions-fr.csv || cp massive-intents/lab-results/predictions-fr.csv .
!test -f timings-fr.csv || cp massive-intents/lab-results/timings-fr.csv .
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import os
import subprocess

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "8", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pull = subprocess.Popen(f"sleep 5 && ollama pull {MODEL}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import pandas as pd
from openai import OpenAI
from pydantic import create_model
from sklearn.metrics import f1_score

train = pd.read_parquet("massive-intents/fr-train.parquet")
intents = pd.read_csv("massive-intents/intents.csv")
test = pd.read_parquet("massive-intents/fr-test.parquet")
# Les textes des demandes, d'après leur id.
predictions = pd.read_csv("predictions-fr.csv", dtype={"id": str}).merge(test[["id", "text"]], on="id")
timings = pd.read_csv("timings-fr.csv")
# 300 demandes de test, tirées au hasard : un LLM est trop lent pour les 2 974.
sample = predictions.sample(300, random_state=0).reset_index(drop=True)
labels = intents.label.tolist()
print(len(labels), "intentions")

pull.wait()  # le modèle est téléchargé
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# Un premier appel charge le modèle sur le GPU, où il reste : avant les calculs en parallèle.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1)
print(MODEL, "prêt")

## 1. Sans exemple

Le schéma de sortie, `Intent`, n'a qu'un champ, `intent`, dont la valeur est forcément l'une des 60 étiquettes : le modèle ne peut pas répondre hors liste.

Écrivez le prompt système : la tâche, puis la liste des intentions avec leur description (`intents.description_fr`). Écrivez `classify(text, system)` qui renvoie l'intention choisie (`client.chat.completions.parse(..., response_format=Intent, temperature=0)`), classez les 300 demandes de `sample` en parallèle (`ThreadPoolExecutor(8)`), et mesurez le F1 macro et le temps par texte.

In [ ]:
Intent = create_model("Intent", intent=(Literal[tuple(labels)], ...))

In [ ]:
# Votre code ici

### Solution

In [ ]:
LIST = "\n".join(f"- {row.label} : {row.description_fr}" for row in intents.itertuples())
ZERO_SHOT = f"""Tu classes les demandes faites à un assistant vocal selon leur intention.
Choisis exactement une intention de cette liste :
{LIST}"""


def classify(text: str, system: str) -> str:
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": text}],
        response_format=Intent,
        temperature=0,
    )
    return response.choices[0].message.parsed.intent


def run(system: str) -> tuple[list[str], float]:
    start = time.time()
    with ThreadPoolExecutor(8) as pool:
        found = list(pool.map(lambda text: classify(text, system), sample.text))
    return found, time.time() - start


zero_shot, zero_seconds = run(ZERO_SHOT)
print(f"F1 macro {f1_score(sample.label, zero_shot, average='macro'):.3f}, {1000 * zero_seconds / len(sample):.0f} ms/texte")

## 2. Avec quelques exemples

Ajoutez au prompt deux exemples d'entraînement par intention (`train.groupby("label").sample(2, random_state=0)`). Le F1 progresse-t-il ? Combien de tokens le prompt compte-t-il maintenant (`response.usage.prompt_tokens`), et qu'est-ce que cela coûterait par une API, à 0,10 € par million de tokens d'entrée, pour 1 million de demandes ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
examples = train.groupby("label").sample(2, random_state=0)
EXAMPLES = "\n".join(f"« {row.text} » -> {row.label}" for row in examples.itertuples())
FEW_SHOT = f"{ZERO_SHOT}\n\nExemples :\n{EXAMPLES}"

few_shot, few_seconds = run(FEW_SHOT)
print(f"F1 macro {f1_score(sample.label, few_shot, average='macro'):.3f}, {1000 * few_seconds / len(sample):.0f} ms/texte")

usage = client.chat.completions.create(
    model=MODEL, messages=[{"role": "system", "content": FEW_SHOT}, {"role": "user", "content": sample.text[0]}], max_tokens=1
).usage
# 1 million de demandes de prompt_tokens tokens, à 0,10 € par million de tokens :
print(usage.prompt_tokens, "tokens de prompt :", f"{usage.prompt_tokens * 0.10:.0f} € pour 1 million de demandes")

# Un long préfixe commun (consignes + exemples) est exactement ce que le cache de prompt
# des API facture à prix réduit.

## 3. Comparer toutes les méthodes

Rassemblez dans un tableau, sur les **mêmes** 300 demandes : le F1 macro de chaque méthode d'hier (colonnes de `sample`) et des deux du jour, leur temps d'entraînement et leur temps par texte (`timings`). Laquelle mettriez-vous en production pour 10 millions de demandes par mois ? Et pour un prototype à livrer demain ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
methods = [column for column in sample.columns if column not in ("id", "label", "text")]
rows = []
for method in methods:
    timing = timings.set_index("method").loc[method]
    rows.append((method, f1_score(sample.label, sample[method], average="macro"), timing.train_s, timing.ms_per_text))
rows.append(("LLM sans exemple", f1_score(sample.label, zero_shot, average="macro"), 0, 1000 * zero_seconds / len(sample)))
rows.append(("LLM avec exemples", f1_score(sample.label, few_shot, average="macro"), 0, 1000 * few_seconds / len(sample)))
pd.DataFrame(rows, columns=["méthode", "F1 macro", "entraînement (s)", "ms/texte"]).round(3)

# En production à gros volume : l'encodeur affiné (le plus juste, quelques ms par texte sur
# CPU). Pour un prototype sans données étiquetées : le LLM, tout de suite.

## 4. Lire les erreurs

Affichez 10 erreurs du meilleur modèle (le texte, l'étiquette attendue, la prédiction). Erreur du modèle, ou étiquette discutable ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
best = max(methods, key=lambda m: f1_score(sample.label, sample[m], average="macro"))
errors = sample[sample[best] != sample.label]
print(best)
errors[["text", "label", best]].head(10)